In [ ]:
# ============================================================================
# Handles: JS redirects, dynamic content, year extraction, deduplication
# ============================================================================

import re
import random
import time
from datetime import datetime
from typing import List, Dict, Any, Optional, Set, Tuple
from dataclasses import dataclass, asdict
from urllib.parse import urljoin, urlparse, unquote, urlunparse
import json
import os
from collections import defaultdict
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException
from bs4 import BeautifulSoup
import logging

from webdriver_manager.chrome import ChromeDriverManager

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

@dataclass
class DocumentInfo:
    """Structure to hold document information"""
    title: str
    url: str
    normalized_url: str
    document_type: str
    publication_date: Optional[str]
    file_extension: str
    relevance_score: float
    section: str
    subsection: str
    content_preview: str
    metadata: Dict[str, Any]
    is_direct_file: bool = False
    extracted_year: Optional[int] = None
    extracted_quarter: Optional[int] = None

class IRDocumentExtractor:
    """Extract IR documents with ALL issues fixed"""
    
    def __init__(self, debug: bool = True, max_sections: int = 20, 
                 request_delay: Tuple[float, float] = (2, 4), max_retries: int = 2):
        self.debug = debug
        self.max_sections = max_sections
        self.request_delay = request_delay
        self.max_retries = max_retries
        self.file_extensions = ['.pdf', '.xlsx', '.xls', '.pptx', '.ppt', 
                                '.docx', '.doc', '.csv', '.mp3', '.mp4']
        self.seen_urls: Set[str] = set()
        self.driver = None
        self.current_year = datetime.now().year
    
    def _setup_selenium(self):
        """Setup Selenium WebDriver"""
        if self.debug:
            logger.info("Setting up ChromeDriver...")
        
        try:
            chrome_options = Options()
            chrome_options.add_argument("--headless=new")
            chrome_options.add_argument("--no-sandbox")
            chrome_options.add_argument("--disable-dev-shm-usage")
            chrome_options.add_argument("--disable-blink-features=AutomationControlled")
            chrome_options.add_argument("--window-size=1920,1080")
            chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
            chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
            chrome_options.add_experimental_option('useAutomationExtension', False)
            chrome_options.add_argument("--disable-extensions")
            chrome_options.add_argument("--disable-gpu")
            chrome_options.page_load_strategy = 'normal'  # Changed from 'eager' to 'normal'
            
            service = Service(ChromeDriverManager().install())
            driver = webdriver.Chrome(service=service, options=chrome_options)
            driver.set_page_load_timeout(45)  # Increased timeout
            
            driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
                'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
            })
            
            if self.debug:
                logger.info("ChromeDriver ready")
            
            return driver
        except Exception as e:
            logger.error(f"Failed to setup ChromeDriver: {e}")
            raise
    
    def _normalize_url(self, url: str) -> str:
        """Normalize URL for deduplication"""
        try:
            parsed = urlparse(url)
            normalized = urlunparse((parsed.scheme, parsed.netloc, parsed.path, 
                                    parsed.params, '', ''))
            return normalized
        except Exception:
            return url
    
    def _is_valid_ir_url(self, url: str) -> bool:
        """Validate that URL is actually an IR page"""
        url_lower = url.lower()
        
        # Invalid patterns
        invalid_patterns = ['smartphone', 'iphone', 'product', 'shop', '/careers']
        if any(bad in url_lower for bad in invalid_patterns):
            return False
        
        # Valid patterns
        valid_patterns = ['investor', 'ir.', 'shareholders', '/investor']
        return any(good in url_lower for good in valid_patterns)
    
    def _is_direct_file(self, url: str) -> bool:
        """Check if URL is direct downloadable file"""
        url_lower = url.lower()
        
        if any(ext in url_lower for ext in self.file_extensions):
            return True
        
        cdn_patterns = [
            r'\.q4cdn\.com/.*/files/', r'\.q4cdn\.com/.*/doc_',
            r'\.cloudfront\.net/', r'/static-files/[a-f0-9\-]{30,}',
            r'/files/doc_', r'/_assets/.*/', r'/doc_financials/',
            r'/doc_downloads/', r'/doc_earnings/'
        ]
        
        return any(re.search(pattern, url_lower) for pattern in cdn_patterns)
    
    def _wait_for_dynamic_content(self, url: str):
        """
        FIX: Wait for dynamic content to load (Disney, IBM, Merck)
        Handles JS redirects and AJAX-loaded content
        """
        try:
            # Wait for body tag first
            WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
            
            # FIX: Wait for JavaScript redirects 
            time.sleep(4)  # Allow 3-4 seconds for redirects
            
            # FIX: Wait for links to appear
            # Try multiple strategies
            try:
                WebDriverWait(self.driver, 8).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='pdf'], a[href*='10-k'], a[href*='10-q']"))
                )
            except TimeoutException:
                # Fallback: wait for any links with 'investor' or 'financial'
                try:
                    WebDriverWait(self.driver, 5).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='financial'], a[href*='report']"))
                    )
                except TimeoutException:
                    # Last resort: just wait for many links
                    WebDriverWait(self.driver, 5).until(
                        lambda d: len(d.find_elements(By.TAG_NAME, "a")) > 20
                    )
            
           
            time.sleep(2)
            
        except TimeoutException:
            logger.warning(f"Timeout waiting for dynamic content")
    
    def extract_documents(self, ir_url: str, ticker: str, 
                         company_name: str) -> List[DocumentInfo]:
        """Extract all documents with retry logic and global deduplication"""
        
        # FIX: Validate IR URL before extraction
        if not self._is_valid_ir_url(ir_url):
            logger.error(f"❌ {ticker}: Invalid IR URL: {ir_url}")
            return []
        
        if self.debug:
            logger.info(f"\n[{ticker}] {company_name}")
            logger.info(f"URL: {ir_url}")
        
        all_raw_documents = []
        
        for attempt in range(self.max_retries):
            try:
                if not self.driver:
                    self.driver = self._setup_selenium()
                
                self.driver.get(ir_url)
                
                # FIX: Wait for dynamic content 
                self._wait_for_dynamic_content(ir_url)
                
                sections = self._discover_sections(ir_url)
                
                if self.debug:
                    logger.info(f"Found {len(sections)} main sections")
                
                for section_name, section_url in sections:
                    if self.debug:
                        logger.info(f"\nExtracting from section: {section_name}")
                    
                    try:
                        self.driver.get(section_url)
                        
                        # FIX: Wait for dynamic content on each section page
                        self._wait_for_dynamic_content(section_url)
                        time.sleep(random.uniform(*self.request_delay))
                        
                        soup = BeautifulSoup(self.driver.page_source, 'html.parser')
                        section_docs = self._extract_all_files_from_page(
                            soup, section_url, section_name
                        )
                        
                        if self.debug:
                            logger.info(f"  Found {len(section_docs)} documents")
                        
                        all_raw_documents.extend(section_docs)
                    
                    except TimeoutException:
                        logger.warning(f"Timeout loading section: {section_name}")
                        continue
                    except Exception as e:
                        logger.error(f"Error processing section {section_name}: {e}")
                        continue
                
                if len(all_raw_documents) == 0:
                    if attempt < self.max_retries - 1:
                        logger.warning(f"No documents found for {ticker}, retry {attempt + 1}/{self.max_retries}")
                        time.sleep(5)
                        continue
                    else:
                        logger.error(f"❌ {ticker}: Failed to extract any documents after {self.max_retries} attempts")
                        return []
                
                break
                
            except Exception as e:
                if attempt < self.max_retries - 1:
                    logger.warning(f"Error for {ticker}, retrying: {e}")
                    time.sleep(5)
                else:
                    logger.error(f"❌ {ticker}: Failed after {self.max_retries} attempts: {e}")
                    return []
        
        all_raw_documents = [doc for doc in all_raw_documents if doc.is_direct_file]
        
        if self.debug:
            logger.info(f"\nTotal raw documents extracted: {len(all_raw_documents)}")
        
        # FIX: Improved global deduplication
        deduplicated_documents = self._global_deduplicate_improved(all_raw_documents)
        
        if self.debug:
            logger.info(f"After global deduplication: {len(deduplicated_documents)} unique latest documents")
            logger.info("\nFinal document list:")
            for doc in deduplicated_documents:
                date_info = f"({doc.extracted_year or '?'}"
                if doc.extracted_quarter:
                    date_info += f" Q{doc.extracted_quarter}"
                date_info += ")"
                logger.info(f"  • {doc.document_type}: {doc.title[:50]}... {date_info}")
        
        return deduplicated_documents
    
    def _global_deduplicate_improved(self, documents: List[DocumentInfo]) -> List[DocumentInfo]:
        """
        IMPROVED: Prefer documents WITH years over documents WITHOUT years
        Flag old documents as errors
        """
        by_type = defaultdict(list)
        for doc in documents:
            by_type[doc.document_type].append(doc)
        
        latest_documents = []
        
        for doc_type, docs in by_type.items():
            # FIX: Separate docs with/without years
            docs_with_year = [d for d in docs if d.extracted_year is not None]
            docs_without_year = [d for d in docs if d.extracted_year is None]
            
            # Prefer documents WITH years
            if docs_with_year:
                sorted_docs = sorted(
                    docs_with_year,
                    key=lambda d: (
                        d.extracted_year,
                        d.extracted_quarter if d.extracted_quarter is not None else -1,
                        d.relevance_score
                    ),
                    reverse=True
                )
                latest = sorted_docs[0]
                
                # FIX: ERROR (not warning) if keeping old document
                if latest.extracted_year < self.current_year - 1:
                    logger.error(f"🚨 {doc_type}: KEEPING OLD DOCUMENT from {latest.extracted_year} (should be {self.current_year} or {self.current_year-1})")
            else:
                # Fallback: use documents without years
                if docs_without_year:
                    latest = docs_without_year[0]
                    logger.warning(f"⚠️ {doc_type}: No documents with extractable years, using first available")
                else:
                    continue
            
            latest_documents.append(latest)
            
            if self.debug and len(docs) > 1:
                logger.info(f"\n  📊 {doc_type}: Found {len(docs)} documents, keeping latest")
                logger.info(f"     ✅ KEPT: {latest.title[:40]} ({latest.extracted_year or 'NO YEAR'})")
                for discarded in sorted_docs[1:3] if docs_with_year else docs_without_year[1:3]:
                    logger.info(f"     ❌ Discarded: {discarded.title[:40]} ({discarded.extracted_year or 'NO YEAR'})")
        
        return latest_documents
    
    def _discover_sections(self, base_url: str) -> List[Tuple[str, str]]:
        """Discover sections with priority scoring"""
        sections = []
        soup = BeautifulSoup(self.driver.page_source, 'html.parser')
        all_links = soup.find_all('a', href=True)
        
        priority_keywords = ['sec filing', '10-k', '10-q', 'financial', 'annual report', 
                            'quarterly', 'earnings', 'investor']
        
        for link in all_links:
            try:
                text = link.get_text(strip=True)
                href = link.get('href', '')
                
                if not href or href.startswith('#') or href.startswith('javascript:'):
                    continue
                
                if not text or len(text) < 3:
                    continue
                
                full_url = urljoin(base_url, href)
                
                if urlparse(full_url).netloc != urlparse(base_url).netloc:
                    continue
                
                if self._is_likely_section_link(text, href, link):
                    priority = sum(1 for kw in priority_keywords 
                                  if kw in text.lower() or kw in href.lower())
                    sections.append((text, full_url, priority))
            except Exception:
                continue
        
        sections.sort(key=lambda x: x[2], reverse=True)
        
        unique_sections = {}
        for name, url, priority in sections:
            normalized = self._normalize_url(url)
            if normalized not in unique_sections and normalized != self._normalize_url(base_url):
                unique_sections[normalized] = (name, url)
        
        result = list(unique_sections.values())[:self.max_sections]
        
        if self.debug and result:
            logger.info("Discovered sections (priority order):")
            for name, _ in result[:10]:
                logger.info(f"  • {name}")
        
        return result
    
    def _is_likely_section_link(self, text: str, href: str, link) -> bool:
        """Determine if link is likely a main section"""
        text_lower = text.lower()
        href_lower = href.lower()
        
        word_count = len(text.split())
        if word_count < 1 or word_count > 6:
            return False
        
        path_depth = len([p for p in urlparse(href_lower).path.split('/') if p])
        if path_depth > 3:
            return False
        
        financial_indicators = [
            'financial', 'investor', 'earning', 'report', 'filing', 'sec',
            'presentation', 'event', 'news', 'press', 'annual', 'quarterly',
            'result', 'document', 'library', 'information'
        ]
        
        has_financial_term = any(term in text_lower or term in href_lower 
                                 for term in financial_indicators)
        
        if not has_financial_term:
            return False
        
        exclude_terms = [
            'contact', 'email alert', 'subscribe', 'rss', 'faq', 'help',
            'cookie', 'privacy', 'terms', 'accessibility', 'sitemap'
        ]
        
        if any(term in text_lower for term in exclude_terms):
            return False
        
        return True
    
    def _extract_all_files_from_page(self, soup: BeautifulSoup, 
                                     base_url: str, section_name: str) -> List[DocumentInfo]:
        """Extract all file links from a page"""
        documents = []
        all_links = soup.find_all('a', href=True)
        
        for link in all_links:
            try:
                href = link.get('href', '')
                if not href or href.startswith('#') or href.startswith('javascript:'):
                    continue
                
                full_url = urljoin(base_url, href)
                normalized_url = self._normalize_url(full_url)
                
                if normalized_url in self.seen_urls:
                    continue
                
                if not self._is_direct_file(full_url):
                    continue
                
                self.seen_urls.add(normalized_url)
                
                text = link.get_text(strip=True)
                title = link.get('title', '')
                
                if not text or len(text) < 5:
                    text = self._extract_filename_from_url(full_url)
                
                doc_type = self._classify_document(full_url, text, title)
                pub_date = self._extract_date(text + ' ' + full_url + ' ' + title)
                year = self._extract_year_improved(text + ' ' + full_url + ' ' + title)
                quarter = self._extract_quarter(text + ' ' + full_url + ' ' + title)
                
                doc = DocumentInfo(
                    title=text[:200],
                    url=full_url,
                    normalized_url=normalized_url,
                    document_type=doc_type,
                    publication_date=pub_date,
                    file_extension=self._get_extension(full_url),
                    relevance_score=100.0,
                    section=section_name,
                    subsection=doc_type,
                    content_preview=text[:150],
                    metadata={'ticker': '', 'company': ''},
                    is_direct_file=True,
                    extracted_year=year,
                    extracted_quarter=quarter
                )
                
                documents.append(doc)
            except Exception:
                continue
        
        return documents
    
    def _classify_document(self, url: str, text: str, title: str) -> str:
        """
        FIXED: Classify with PRIORITY ORDER (most specific first)
        """
        combined = f"{url} {text} {title}".lower()
        
        classifications = [
            ('10-K Annual Report', ['10-k', 'form 10-k', 'form10-k', '10k annual']),
            ('10-Q Quarterly Report', ['10-q', 'form 10-q', 'form10-q', '10q quarter']),
            ('8-K Current Report', ['8-k', 'form 8-k', 'form8-k']),
            ('Proxy Statement', ['proxy statement', 'proxy', 'def 14a', 'def14a']),
            ('Press Release', ['press release', 'earnings release']),
            ('CFO Commentary', ['cfo commentary', 'cfo comment']),
            ('Audio Webcast', ['webcast', 'audio', '.mp3', 'call.mp3']),
            ('Transcript', ['transcript', 'call transcript', 'earnings call']),
            ('Presentation', ['presentation', 'slides', 'deck', 'investor deck']),
            ('Revenue Trend', ['revenue trend', 'quarterly revenue']),
            ('Supplemental Data', ['supplemental', 'data table', 'supplemental data']),
            ('Annual Report', ['annual report']),
            ('Excel Data', ['.xlsx', '.xls']),
        ]
        
        for doc_type, keywords in classifications:
            if any(keyword in combined for keyword in keywords):
                return doc_type
        
        return 'PDF Document' if '.pdf' in url else 'Financial Document'
    
    def _extract_filename_from_url(self, url: str) -> str:
        """Extract clean filename from URL"""
        try:
            parsed = urlparse(url)
            filename = os.path.basename(unquote(parsed.path))
            filename = re.sub(r'[-_]', ' ', filename)
            filename = re.sub(r'\.(pdf|xlsx?|pptx?|docx?)$', '', filename, flags=re.I)
            # FIX: Only remove long hashes (32+ chars), keep meaningful codes
            filename = re.sub(r'\b[a-f0-9]{32,}\b', '', filename, flags=re.I)
            filename = re.sub(r'\s+', ' ', filename).strip()
            return filename or 'Financial Document'
        except Exception:
            return 'Financial Document'
    
    def _extract_date(self, text: str) -> Optional[str]:
        """Extract date from text"""
        patterns = [
            r'Q([1-4])\s*20([12]\d)',
            r'20([12]\d)\s*Q([1-4])',
            r'FY\s*(\d{2})',
            r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d{1,2},?\s+20[12]\d',
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.I)
            if match:
                return match.group(0)
        return None
    
    def _extract_year_improved(self, text: str) -> Optional[int]:
        """
        IMPROVED: Extract year with 6 strategies
        Handles fiscal years, SEC filings, short years
        """
        # Strategy 1: Standard 20XX format
        year_match = re.search(r'20([12]\d)', text)
        if year_match:
            year = int(year_match.group(0))
            if 2020 <= year <= self.current_year + 2:
                return year
        
        # Strategy 2: Fiscal year format (FY25, FY2025, fy25)
        fy_match = re.search(r'fy\s*(\d{2,4})', text, re.I)
        if fy_match:
            fy_year = fy_match.group(1)
            if len(fy_year) == 2:
                year = int(fy_year)
                full_year = 2000 + year
                if 2020 <= full_year <= self.current_year + 2:
                    return full_year
            elif len(fy_year) == 4:
                year = int(fy_year)
                if 2020 <= year <= self.current_year + 2:
                    return year
        
        # Strategy 3: Extract from URL path (/2025/, /q2/2025/)
        url_year = re.search(r'/(\d{4})/', text)
        if url_year:
            year = int(url_year.group(1))
            if 2020 <= year <= self.current_year + 2:
                return year
        
        # Strategy 4: Year in filename (report-2025.pdf)
        filename_year = re.search(r'[-_](\d{4})[.-_]', text)
        if filename_year:
            year = int(filename_year.group(1))
            if 2020 <= year <= self.current_year + 2:
                return year
        
        # NEW Strategy 5: SEC filing numbers (25-000063 = 2025)
        sec_filing = re.search(r'(\d{2})-\d{6}', text)
        if sec_filing:
            filing_year = int(sec_filing.group(1))
            full_year = 2000 + filing_year
            if 2020 <= full_year <= self.current_year + 2:
                return full_year
        
        # NEW Strategy 6: Short year at end (4Q 17 = 2017, 10K 4Q 23 = 2023)
        short_year_patterns = [
            r'(?:4q|q4)\s*(\d{2})\b',  # "4Q 17", "Q4 23"
            r'\b(\d{2})(?=\s*$)',  # "2 digits at end"
        ]
        for pattern in short_year_patterns:
            short_match = re.search(pattern, text, re.I)
            if short_match:
                year_num = int(short_match.group(1))
                if 15 <= year_num <= 30:  # 2015-2030
                    full_year = 2000 + year_num
                    if 2020 <= full_year <= self.current_year + 2:
                        return full_year
        
        return None
    
    def _extract_quarter(self, text: str) -> Optional[int]:
        """Extract quarter from text"""
        quarter_match = re.search(r'[q\-]([1-4])(?:\s|q|$)', text, re.I)
        return int(quarter_match.group(1)) if quarter_match else None
    
    def _get_extension(self, url: str) -> str:
        """Get file extension"""
        url_lower = url.lower()
        for ext in self.file_extensions:
            if ext in url_lower:
                return ext
        return '.unknown'
    
    def close(self):
        """Close WebDriver"""
        if self.driver:
            try:
                self.driver.quit()
            except Exception:
                pass
            self.driver = None


# ============================================================================
# MAIN EXECUTION FUNCTIONS
# ============================================================================

def extract_all_companies(input_json_path: str, output_json_path: str = None,
                         start_index: int = 0, end_index: int = None,
                         debug: bool = True) -> Dict[str, List[DocumentInfo]]:
    """Extract documents for all companies"""
    print("\n" + "="*80)
    print("IR DOCUMENT EXTRACTION - ALL FIXES APPLIED")
    print("✅ JS redirects • Dynamic content • Improved year extraction • Better dedup")
    print("="*80)
    
    with open(input_json_path, 'r') as f:
        companies = json.load(f)
    
    if end_index is not None:
        companies = companies[start_index:end_index]
    else:
        companies = companies[start_index:]
    
    print(f"Processing {len(companies)} companies (index {start_index} to {start_index + len(companies)})")
    
    extractor = IRDocumentExtractor(debug=debug)
    all_documents = {}
    
    try:
        for i, company in enumerate(companies, 1):
            ticker = company.get('ticker', 'UNKNOWN')
            company_name = company.get('company_name', 'Unknown Company')
            ir_url = company.get('investor_relations_url', '')
            
            if not ir_url:
                logger.warning(f"No IR URL for {ticker}, skipping...")
                all_documents[ticker] = []
                continue
            
            print(f"\n[{i}/{len(companies)}] {ticker}")
            
            try:
                documents = extractor.extract_documents(ir_url, ticker, company_name)
                
                for doc in documents:
                    doc.metadata['ticker'] = ticker
                    doc.metadata['company'] = company_name
                
                all_documents[ticker] = documents
                time.sleep(random.uniform(2, 4))
                
            except Exception as e:
                logger.error(f"Error processing {ticker}: {e}")
                all_documents[ticker] = []
                continue
    
    finally:
        extractor.close()
    
    # Print summary
    total = sum(len(docs) for docs in all_documents.values())
    with_docs = sum(1 for docs in all_documents.values() if docs)
    total_pdfs = sum(sum(1 for d in docs if '.pdf' in d.file_extension) 
                     for docs in all_documents.values())
    total_excel = sum(sum(1 for d in docs if '.xls' in d.file_extension) 
                      for docs in all_documents.values())
    
    # Count documents with old years
    old_docs = sum(sum(1 for d in docs if d.extracted_year and d.extracted_year < datetime.now().year - 1) 
                   for docs in all_documents.values())
    
    print("\n" + "="*80)
    print("EXTRACTION SUMMARY")
    print("="*80)
    print(f"✅ Total unique documents: {total}")
    print(f"✅ PDF files: {total_pdfs}")
    print(f"✅ Excel files: {total_excel}")
    print(f"✅ Companies with documents: {with_docs}/{len(companies)}")
    print(f"⚠️ Companies with NO documents: {len(companies) - with_docs}")
    print(f"🚨 Documents with old years (<2024): {old_docs}")
    if companies:
        print(f"✅ Average per company: {total/len(companies):.1f}")
    
    if output_json_path:
        export_data = {}
        for ticker, documents in all_documents.items():
            export_data[ticker] = [asdict(doc) for doc in documents]
        
        with open(output_json_path, 'w') as f:
            json.dump(export_data, f, indent=2, default=str)
        
        print(f"\n✅ Exported to {output_json_path}")
    
    return all_documents


def extract_in_batches(input_json_path: str, output_dir: str = "output",
                      batch_size: int = 10, debug: bool = True):
    """Extract documents in batches"""
    with open(input_json_path, 'r') as f:
        companies = json.load(f)
    
    total_companies = len(companies)
    num_batches = (total_companies + batch_size - 1) // batch_size
    
    print(f"\n{'='*80}")
    print(f"BATCH EXTRACTION: {total_companies} companies in {num_batches} batches")
    print(f"{'='*80}")
    
    os.makedirs(output_dir, exist_ok=True)
    all_results = {}
    
    for batch_num in range(num_batches):
        start_idx = batch_num * batch_size
        end_idx = min(start_idx + batch_size, total_companies)
        
        print(f"\n{'='*80}")
        print(f"BATCH {batch_num + 1}/{num_batches}: Companies {start_idx} to {end_idx}")
        print(f"{'='*80}")
        
        batch_output = os.path.join(output_dir, f"batch_{batch_num + 1:03d}.json")
        
        batch_results = extract_all_companies(
            input_json_path,
            output_json_path=batch_output,
            start_index=start_idx,
            end_index=end_idx,
            debug=debug
        )
        
        all_results.update(batch_results)
        print(f"\n✅ Batch {batch_num + 1} complete. Saved to {batch_output}")
    
    combined_output = os.path.join(output_dir, "all_companies_combined.json")
    export_data = {}
    for ticker, documents in all_results.items():
        export_data[ticker] = [asdict(doc) for doc in documents]
    
    with open(combined_output, 'w') as f:
        json.dump(export_data, f, indent=2, default=str)
    
    print(f"\n{'='*80}")
    print(f"✅ ALL BATCHES COMPLETE")
    print(f"✅ Combined results saved to {combined_output}")
    print(f"{'='*80}")
    
    return all_results


# ============================================================================
# USAGE EXAMPLES
# ============================================================================

"""
# Example 1: Quick test with first 5 companies
results = extract_all_companies(
    input_json_path='cnbc_companies_with_ir.json',
    output_json_path='test_fully_corrected.json',
    start_index=0,
    end_index=5,
    debug=True
)

# Example 2: Extract all companies
all_results = extract_all_companies(
    input_json_path='cnbc_companies_with_ir.json',
    output_json_path='all_documents_fully_corrected.json',
    debug=True
)

# Example 3: Batch processing (recommended for large datasets)
batch_results = extract_in_batches(
    input_json_path='cnbc_companies_with_ir.json',
    output_dir='fully_corrected_output',
    batch_size=20,
    debug=True
)
"""

"\n# Example 1: Quick test with first 5 companies\nresults = extract_all_companies(\n    input_json_path='cnbc_companies_with_ir.json',\n    output_json_path='test_fully_corrected.json',\n    start_index=0,\n    end_index=5,\n    debug=True\n)\n\n# Example 2: Extract all companies\nall_results = extract_all_companies(\n    input_json_path='cnbc_companies_with_ir.json',\n    output_json_path='all_documents_fully_corrected.json',\n    debug=True\n)\n\n# Example 3: Batch processing (recommended for large datasets)\nbatch_results = extract_in_batches(\n    input_json_path='cnbc_companies_with_ir.json',\n    output_dir='fully_corrected_output',\n    batch_size=20,\n    debug=True\n)\n"

In [7]:
all_results = extract_all_companies(
    input_json_path='../data/catalogue/cnbc_companies_with_ir.json',
    output_json_path='../data/documents/all_documents_fully_corrected.json',
    debug=True
)

2025-10-09 01:53:35,679 - INFO - 
[AMGN] AMGN
2025-10-09 01:53:35,680 - INFO - URL: http://investors.amgen.com/
2025-10-09 01:53:35,680 - INFO - Setting up ChromeDriver...
2025-10-09 01:53:35,681 - INFO - ====== WebDriver manager ======



IR DOCUMENT EXTRACTION - ALL FIXES APPLIED
✅ JS redirects • Dynamic content • Improved year extraction • Better dedup
Processing 30 companies (index 0 to 30)

[1/30] AMGN


2025-10-09 01:53:35,985 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-09 01:53:36,112 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-09 01:53:36,203 - INFO - Driver [/Users/RiyanshiKedia/.wdm/drivers/chromedriver/mac64/141.0.7390.65/chromedriver-mac-arm64/chromedriver] found in cache
2025-10-09 01:53:37,675 - INFO - ChromeDriver ready
2025-10-09 01:53:46,189 - INFO - Discovered sections (priority order):
2025-10-09 01:53:46,190 - INFO -   • Quarterly Earnings
2025-10-09 01:53:46,190 - INFO -   • SEC Filings
2025-10-09 01:53:46,190 - INFO -   • Annual Reports
2025-10-09 01:53:46,191 - INFO -   • Amgen Q3 2025 Earnings Conference Call
2025-10-09 01:53:46,191 - INFO -   • Tax Forms
2025-10-09 01:53:46,192 - INFO -   • Reconciliations
2025-10-09 01:53:46,193 - INFO -   • Amgen Green Financing Framework 2022
2025-10-09 01:53:46,193 - INFO -   • Amgen Q2 2025 Earnings Conference Call
2025-10-09 01:53:46,193 - INFO -   • Q2 2025 Earnings Presentation
2


[2/30] AMZN


2025-10-09 01:57:09,471 - INFO - Discovered sections (priority order):
2025-10-09 01:57:09,472 - INFO -   • Investor Relations
2025-10-09 01:57:09,472 - INFO -   • Annual reports, proxies and shareholder letters
2025-10-09 01:57:09,472 - INFO -   • Quarterly results
2025-10-09 01:57:09,473 - INFO -   • SEC filings
2025-10-09 01:57:09,473 - INFO -   • Investors
2025-10-09 01:57:09,473 - INFO -   • Events
2025-10-09 01:57:09,474 - INFO - Found 6 main sections
2025-10-09 01:57:09,474 - INFO - 
Extracting from section: Investor Relations
2025-10-09 01:57:17,638 - INFO -   Found 1 documents
2025-10-09 01:57:17,638 - INFO - 
Extracting from section: Annual reports, proxies and shareholder letters
2025-10-09 01:57:26,536 - INFO -   Found 87 documents
2025-10-09 01:57:26,536 - INFO - 
Extracting from section: Quarterly results
2025-10-09 01:57:35,543 - INFO -   Found 79 documents
2025-10-09 01:57:35,544 - INFO - 
Extracting from section: SEC filings
2025-10-09 01:57:45,401 - INFO -   Found 187

...


[29/30] NKE


2025-10-09 03:21:45,999 - INFO - Discovered sections (priority order):
2025-10-09 03:21:46,000 - INFO -   • Quarterly Earnings
2025-10-09 03:21:46,000 - INFO -   • NYSE NKE  $69.09 +0.18
2025-10-09 03:21:46,000 - INFO -   • Investor News
2025-10-09 03:21:46,000 - INFO -   • Direct Investment
2025-10-09 03:21:46,001 - INFO -   • Board of Directors
2025-10-09 03:21:46,001 - INFO -   • Learn More
2025-10-09 03:21:46,002 - INFO - Found 6 main sections
2025-10-09 03:21:46,002 - INFO - 
Extracting from section: Quarterly Earnings
2025-10-09 03:21:54,970 - INFO -   Found 180 documents
2025-10-09 03:21:54,970 - INFO - 
Extracting from section: NYSE NKE  $69.09 +0.18
2025-10-09 03:22:12,872 - INFO -   Found 0 documents
2025-10-09 03:22:12,873 - INFO - 
Extracting from section: Investor News
2025-10-09 03:22:23,366 - INFO -   Found 0 documents
2025-10-09 03:22:23,367 - INFO - 
Extracting from section: Direct Investment
2025-10-09 03:22:33,684 - INFO -   Found 3 documents
2025-10-09 03:22:33,685 


[30/30] PG


2025-10-09 03:23:09,773 - INFO - Discovered sections (priority order):
2025-10-09 03:23:09,774 - INFO -   • Annual Reports
2025-10-09 03:23:09,774 - INFO -   • SEC Filings
2025-10-09 03:23:09,774 - INFO -   • Financials
2025-10-09 03:23:09,774 - INFO -   • Overview
2025-10-09 03:23:09,774 - INFO -   • About P&G
2025-10-09 03:23:09,775 - INFO -   • Company Strategy
2025-10-09 03:23:09,775 - INFO -   • News
2025-10-09 03:23:09,775 - INFO -   • Events & Presentations
2025-10-09 03:23:09,775 - INFO -   • Stock Quote
2025-10-09 03:23:09,776 - INFO -   • Dividend History
2025-10-09 03:23:09,776 - INFO - Found 14 main sections
2025-10-09 03:23:09,776 - INFO - 
Extracting from section: Annual Reports
2025-10-09 03:23:19,150 - INFO -   Found 29 documents
2025-10-09 03:23:19,150 - INFO - 
Extracting from section: SEC Filings
2025-10-09 03:23:28,868 - INFO -   Found 310 documents
2025-10-09 03:23:28,869 - INFO - 
Extracting from section: Financials
2025-10-09 03:23:38,373 - INFO -   Found 1 docum


EXTRACTION SUMMARY
✅ Total unique documents: 194
✅ PDF files: 153
✅ Excel files: 19
✅ Companies with documents: 24/30
⚠️ Companies with NO documents: 6
🚨 Documents with old years (<2024): 11
✅ Average per company: 6.5

✅ Exported to ../data/documents/all_documents_fully_corrected.json


In [50]:
# ============================================================================
# FIXED: Hex string detection + Latest quarterly report identification
# ============================================================================

import re
import random
import time
from datetime import datetime
from typing import List, Dict, Any, Optional, Set, Tuple
from dataclasses import dataclass, asdict
from urllib.parse import urljoin, urlparse, unquote, urlunparse
import json
import os
from collections import defaultdict
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException
from bs4 import BeautifulSoup
import logging

from webdriver_manager.chrome import ChromeDriverManager

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

@dataclass
class DocumentInfo:
    """Structure to hold document information"""
    title: str
    url: str
    normalized_url: str
    document_type: str
    publication_date: Optional[str]
    file_extension: str
    relevance_score: float
    section: str
    subsection: str
    content_preview: str
    metadata: Dict[str, Any]
    is_direct_file: bool = False
    extracted_year: Optional[int] = None
    extracted_quarter: Optional[int] = None

class IRDocumentExtractor:
    """Extract IR documents with ALL issues fixed"""
    
    def __init__(self, debug: bool = True, max_sections: int = 20, 
                 request_delay: Tuple[float, float] = (2, 4), max_retries: int = 2):
        self.debug = debug
        self.max_sections = max_sections
        self.request_delay = request_delay
        self.max_retries = max_retries
        self.file_extensions = ['.pdf', '.xlsx', '.xls', '.pptx', '.ppt', 
                                '.docx', '.doc', '.csv', '.mp3', '.mp4']
        self.seen_urls: Set[str] = set()
        self.driver = None
        self.current_year = datetime.now().year
    
    def _setup_selenium(self):
        """Setup Selenium WebDriver"""
        if self.debug:
            logger.info("Setting up ChromeDriver...")
        
        try:
            chrome_options = Options()
            chrome_options.add_argument("--headless=new")
            chrome_options.add_argument("--no-sandbox")
            chrome_options.add_argument("--disable-dev-shm-usage")
            chrome_options.add_argument("--disable-blink-features=AutomationControlled")
            chrome_options.add_argument("--window-size=1920,1080")
            chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
            chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
            chrome_options.add_experimental_option('useAutomationExtension', False)
            chrome_options.add_argument("--disable-extensions")
            chrome_options.add_argument("--disable-gpu")
            chrome_options.page_load_strategy = 'normal'
            
            service = Service(ChromeDriverManager().install())
            driver = webdriver.Chrome(service=service, options=chrome_options)
            driver.set_page_load_timeout(45)
            
            driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
                'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
            })
            
            if self.debug:
                logger.info("ChromeDriver ready")
            
            return driver
        except Exception as e:
            logger.error(f"Failed to setup ChromeDriver: {e}")
            raise
    
    def _normalize_url(self, url: str) -> str:
        """Normalize URL for deduplication"""
        try:
            parsed = urlparse(url)
            normalized = urlunparse((parsed.scheme, parsed.netloc, parsed.path, 
                                    parsed.params, '', ''))
            return normalized
        except Exception:
            return url
    
    def _is_valid_ir_url(self, url: str) -> bool:
        """Validate that URL is actually an IR page"""
        url_lower = url.lower()
        
        invalid_patterns = ['smartphone', 'iphone', 'product', 'shop', '/careers']
        if any(bad in url_lower for bad in invalid_patterns):
            return False
        
        valid_patterns = ['investor', 'ir.', 'shareholders', '/investor']
        return any(good in url_lower for good in valid_patterns)
    
    def _is_direct_file(self, url: str) -> bool:
        """Check if URL is direct downloadable file"""
        url_lower = url.lower()
        
        if any(ext in url_lower for ext in self.file_extensions):
            return True
        
        cdn_patterns = [
            r'\.q4cdn\.com/.*/files/', r'\.q4cdn\.com/.*/doc_',
            r'\.cloudfront\.net/', r'/static-files/[a-f0-9\-]{30,}',
            r'/files/doc_', r'/_assets/.*/', r'/doc_financials/',
            r'/doc_downloads/', r'/doc_earnings/'
        ]
        
        return any(re.search(pattern, url_lower) for pattern in cdn_patterns)
    
    def _wait_for_dynamic_content(self, url: str):
        """Wait for dynamic content to load"""
        try:
            WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
            
            time.sleep(4)
            
            try:
                WebDriverWait(self.driver, 8).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='pdf'], a[href*='10-k'], a[href*='10-q']"))
                )
            except TimeoutException:
                try:
                    WebDriverWait(self.driver, 5).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='financial'], a[href*='report']"))
                    )
                except TimeoutException:
                    WebDriverWait(self.driver, 5).until(
                        lambda d: len(d.find_elements(By.TAG_NAME, "a")) > 20
                    )
            
            time.sleep(2)
            
        except TimeoutException:
            logger.warning(f"Timeout waiting for dynamic content")
    
    def extract_documents(self, ir_url: str, ticker: str, 
                         company_name: str) -> List[DocumentInfo]:
        """Extract all documents with retry logic and global deduplication"""
        
        if not self._is_valid_ir_url(ir_url):
            logger.error(f"❌ {ticker}: Invalid IR URL: {ir_url}")
            return []
        
        if self.debug:
            logger.info(f"\n[{ticker}] {company_name}")
            logger.info(f"URL: {ir_url}")
        
        all_raw_documents = []
        
        for attempt in range(self.max_retries):
            try:
                if not self.driver:
                    self.driver = self._setup_selenium()
                
                self.driver.get(ir_url)
                self._wait_for_dynamic_content(ir_url)
                
                sections = self._discover_sections(ir_url)
                
                if self.debug:
                    logger.info(f"Found {len(sections)} main sections")
                
                for section_name, section_url in sections:
                    if self.debug:
                        logger.info(f"\nExtracting from section: {section_name}")
                    
                    try:
                        self.driver.get(section_url)
                        self._wait_for_dynamic_content(section_url)
                        time.sleep(random.uniform(*self.request_delay))
                        
                        soup = BeautifulSoup(self.driver.page_source, 'html.parser')
                        section_docs = self._extract_all_files_from_page(
                            soup, section_url, section_name
                        )
                        
                        if self.debug:
                            logger.info(f"  Found {len(section_docs)} documents")
                        
                        all_raw_documents.extend(section_docs)
                    
                    except TimeoutException:
                        logger.warning(f"Timeout loading section: {section_name}")
                        continue
                    except Exception as e:
                        logger.error(f"Error processing section {section_name}: {e}")
                        continue
                
                if len(all_raw_documents) == 0:
                    if attempt < self.max_retries - 1:
                        logger.warning(f"No documents found for {ticker}, retry {attempt + 1}/{self.max_retries}")
                        time.sleep(5)
                        continue
                    else:
                        logger.error(f"❌ {ticker}: Failed to extract any documents after {self.max_retries} attempts")
                        return []
                
                break
                
            except Exception as e:
                if attempt < self.max_retries - 1:
                    logger.warning(f"Error for {ticker}, retrying: {e}")
                    time.sleep(5)
                else:
                    logger.error(f"❌ {ticker}: Failed after {self.max_retries} attempts: {e}")
                    return []
        
        all_raw_documents = [doc for doc in all_raw_documents if doc.is_direct_file]
        
        if self.debug:
            logger.info(f"\nTotal raw documents extracted: {len(all_raw_documents)}")
        
        deduplicated_documents = self._global_deduplicate_improved(all_raw_documents)
        
        # NEW: Identify single latest quarterly report
        latest_quarterly = self.identify_latest_quarterly_report(deduplicated_documents)
        if latest_quarterly and self.debug:
            q_info = f"Q{latest_quarterly.extracted_quarter}" if latest_quarterly.extracted_quarter else ""
            logger.info(f"📊 LATEST QUARTERLY: {latest_quarterly.document_type} - "
                       f"{latest_quarterly.title[:50]}... ({latest_quarterly.extracted_year} {q_info})")
        
        if self.debug:
            logger.info(f"After global deduplication: {len(deduplicated_documents)} unique latest documents")
            logger.info("\nFinal document list:")
            for doc in deduplicated_documents:
                date_info = f"({doc.extracted_year or '?'}"
                if doc.extracted_quarter:
                    date_info += f" Q{doc.extracted_quarter}"
                date_info += ")"
                logger.info(f"  • {doc.document_type}: {doc.title[:50]}... {date_info}")
        
        return deduplicated_documents
    
    def _global_deduplicate_improved(self, documents: List[DocumentInfo]) -> List[DocumentInfo]:
        """Prefer documents WITH years over documents WITHOUT years"""
        by_type = defaultdict(list)
        for doc in documents:
            by_type[doc.document_type].append(doc)
        
        latest_documents = []
        
        for doc_type, docs in by_type.items():
            docs_with_year = [d for d in docs if d.extracted_year is not None]
            docs_without_year = [d for d in docs if d.extracted_year is None]
            
            if docs_with_year:
                sorted_docs = sorted(
                    docs_with_year,
                    key=lambda d: (
                        d.extracted_year,
                        d.extracted_quarter if d.extracted_quarter is not None else -1,
                        d.relevance_score
                    ),
                    reverse=True
                )
                latest = sorted_docs[0]
                
                if latest.extracted_year < self.current_year - 1:
                    logger.error(f"🚨 {doc_type}: KEEPING OLD DOCUMENT from {latest.extracted_year} (should be {self.current_year} or {self.current_year-1})")
            else:
                if docs_without_year:
                    latest = docs_without_year[0]
                    logger.warning(f"⚠️ {doc_type}: No documents with extractable years, using first available")
                else:
                    continue
            
            latest_documents.append(latest)
            
            if self.debug and len(docs) > 1:
                logger.info(f"\n  📊 {doc_type}: Found {len(docs)} documents, keeping latest")
                logger.info(f"     ✅ KEPT: {latest.title[:40]} ({latest.extracted_year or 'NO YEAR'})")
                for discarded in sorted_docs[1:3] if docs_with_year else docs_without_year[1:3]:
                    logger.info(f"     ❌ Discarded: {discarded.title[:40]} ({discarded.extracted_year or 'NO YEAR'})")
        
        return latest_documents
    
    def identify_latest_quarterly_report(self, documents: List[DocumentInfo]) -> Optional[DocumentInfo]:
        """
        NEW: Identify THE single latest quarterly earnings report
        Priority: Press Release > 10-Q > Presentation > Transcript
        """
        if not documents:
            return None
        
        quarterly_types = [
            'Press Release',
            '10-Q Quarterly Report',
            'Presentation',
            'CFO Commentary',
            'Transcript',
            'Excel Data'
        ]
        
        # Prefer docs with quarters
        quarterly_docs = [d for d in documents 
                         if d.document_type in quarterly_types and d.extracted_quarter]
        
        # Fallback: quarterly types without quarter info
        if not quarterly_docs:
            quarterly_docs = [d for d in documents if d.document_type in quarterly_types]
        
        if not quarterly_docs:
            return None
        
        # Type priority (lower = higher priority)
        type_priority = {
            'Press Release': 1,
            '10-Q Quarterly Report': 2,
            'Presentation': 3,
            'CFO Commentary': 4,
            'Transcript': 5,
            'Excel Data': 6
        }
        
        # Sort: newest year > highest quarter > best type > highest score
        sorted_docs = sorted(
            quarterly_docs,
            key=lambda d: (
                d.extracted_year if d.extracted_year else 0,
                d.extracted_quarter if d.extracted_quarter else 0,
                -type_priority.get(d.document_type, 99),
                d.relevance_score
            ),
            reverse=True
        )
        
        return sorted_docs[0]
    
    def _discover_sections(self, base_url: str) -> List[Tuple[str, str]]:
        """Discover sections with priority scoring"""
        sections = []
        soup = BeautifulSoup(self.driver.page_source, 'html.parser')
        all_links = soup.find_all('a', href=True)
        
        priority_keywords = ['sec filing', '10-k', '10-q', 'financial', 'annual report', 
                            'quarterly', 'earnings', 'investor']
        
        for link in all_links:
            try:
                text = link.get_text(strip=True)
                href = link.get('href', '')
                
                if not href or href.startswith('#') or href.startswith('javascript:'):
                    continue
                
                if not text or len(text) < 3:
                    continue
                
                full_url = urljoin(base_url, href)
                
                if urlparse(full_url).netloc != urlparse(base_url).netloc:
                    continue
                
                if self._is_likely_section_link(text, href, link):
                    priority = sum(1 for kw in priority_keywords 
                                  if kw in text.lower() or kw in href.lower())
                    sections.append((text, full_url, priority))
            except Exception:
                continue
        
        sections.sort(key=lambda x: x[2], reverse=True)
        
        unique_sections = {}
        for name, url, priority in sections:
            normalized = self._normalize_url(url)
            if normalized not in unique_sections and normalized != self._normalize_url(base_url):
                unique_sections[normalized] = (name, url)
        
        result = list(unique_sections.values())[:self.max_sections]
        
        if self.debug and result:
            logger.info("Discovered sections (priority order):")
            for name, _ in result[:10]:
                logger.info(f"  • {name}")
        
        return result
    
    def _is_likely_section_link(self, text: str, href: str, link) -> bool:
        """Determine if link is likely a main section"""
        text_lower = text.lower()
        href_lower = href.lower()
        
        word_count = len(text.split())
        if word_count < 1 or word_count > 6:
            return False
        
        path_depth = len([p for p in urlparse(href_lower).path.split('/') if p])
        if path_depth > 3:
            return False
        
        financial_indicators = [
            'financial', 'investor', 'earning', 'report', 'filing', 'sec',
            'presentation', 'event', 'news', 'press', 'annual', 'quarterly',
            'result', 'document', 'library', 'information'
        ]
        
        has_financial_term = any(term in text_lower or term in href_lower 
                                 for term in financial_indicators)
        
        if not has_financial_term:
            return False
        
        exclude_terms = [
            'contact', 'email alert', 'subscribe', 'rss', 'faq', 'help',
            'cookie', 'privacy', 'terms', 'accessibility', 'sitemap'
        ]
        
        if any(term in text_lower for term in exclude_terms):
            return False
        
        return True
    
    def _extract_all_files_from_page(self, soup: BeautifulSoup, 
                                     base_url: str, section_name: str) -> List[DocumentInfo]:
        """Extract all file links from a page"""
        documents = []
        all_links = soup.find_all('a', href=True)
        
        for link in all_links:
            try:
                href = link.get('href', '')
                if not href or href.startswith('#') or href.startswith('javascript:'):
                    continue
                
                full_url = urljoin(base_url, href)
                normalized_url = self._normalize_url(full_url)
                
                if normalized_url in self.seen_urls:
                    continue
                
                if not self._is_direct_file(full_url):
                    continue
                
                self.seen_urls.add(normalized_url)
                
                text = link.get_text(strip=True)
                title = link.get('title', '')
                
                if not text or len(text) < 5:
                    text = self._extract_filename_from_url(full_url)
                
                doc_type = self._classify_document(full_url, text, title)
                pub_date = self._extract_date(text + ' ' + full_url + ' ' + title)
                year = self._extract_year_improved(text + ' ' + full_url + ' ' + title)
                quarter = self._extract_quarter(text + ' ' + full_url + ' ' + title)
                
                doc = DocumentInfo(
                    title=text[:200],
                    url=full_url,
                    normalized_url=normalized_url,
                    document_type=doc_type,
                    publication_date=pub_date,
                    file_extension=self._get_extension(full_url),
                    relevance_score=100.0,
                    section=section_name,
                    subsection=doc_type,
                    content_preview=text[:150],
                    metadata={'ticker': '', 'company': ''},
                    is_direct_file=True,
                    extracted_year=year,
                    extracted_quarter=quarter
                )
                
                documents.append(doc)
            except Exception:
                continue
        
        return documents
    
    def _classify_document(self, url: str, text: str, title: str) -> str:
        """Classify with PRIORITY ORDER"""
        combined = f"{url} {text} {title}".lower()
        
        classifications = [
            ('10-K Annual Report', ['10-k', 'form 10-k', 'form10-k', '10k annual']),
            ('10-Q Quarterly Report', ['10-q', 'form 10-q', 'form10-q', '10q quarter']),
            ('8-K Current Report', ['8-k', 'form 8-k', 'form8-k']),
            ('Proxy Statement', ['proxy statement', 'proxy', 'def 14a', 'def14a']),
            ('Press Release', ['press release', 'earnings release']),
            ('CFO Commentary', ['cfo commentary', 'cfo comment']),
            ('Audio Webcast', ['webcast', 'audio', '.mp3', 'call.mp3']),
            ('Transcript', ['transcript', 'call transcript', 'earnings call']),
            ('Presentation', ['presentation', 'slides', 'deck', 'investor deck']),
            ('Revenue Trend', ['revenue trend', 'quarterly revenue']),
            ('Supplemental Data', ['supplemental', 'data table', 'supplemental data']),
            ('Annual Report', ['annual report']),
            ('Excel Data', ['.xlsx', '.xls']),
        ]
        
        for doc_type, keywords in classifications:
            if any(keyword in combined for keyword in keywords):
                return doc_type
        
        return 'PDF Document' if '.pdf' in url else 'Financial Document'
    
    def _extract_filename_from_url(self, url: str) -> str:
        """Extract clean filename from URL"""
        try:
            parsed = urlparse(url)
            filename = os.path.basename(unquote(parsed.path))
            filename = re.sub(r'[-_]', ' ', filename)
            filename = re.sub(r'\.(pdf|xlsx?|pptx?|docx?)$', '', filename, flags=re.I)
            filename = re.sub(r'\b[a-f0-9]{32,}\b', '', filename, flags=re.I)
            filename = re.sub(r'\s+', ' ', filename).strip()
            return filename or 'Financial Document'
        except Exception:
            return 'Financial Document'
    
    def _extract_date(self, text: str) -> Optional[str]:
        """Extract date from text"""
        patterns = [
            r'Q([1-4])\s*20([12]\d)',
            r'20([12]\d)\s*Q([1-4])',
            r'FY\s*(\d{2})',
            r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d{1,2},?\s+20[12]\d',
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.I)
            if match:
                return match.group(0)
        return None
    
    def _is_valid_year_context(self, year: int, context: str) -> bool:
        """
        NEW: Check if year is in a hex string context
        """
        if not (2015 <= year <= self.current_year):
            return False
        
        # Count hex characters (a-f) in context
        hex_chars = len(re.findall(r'[a-f]', context.lower()))
        total_chars = len(context)
        
        # If >40% hex chars, it's likely a UUID/hash
        if total_chars > 0 and (hex_chars / total_chars) > 0.4:
            return False
        
        return True
    
    def _extract_year_improved(self, text: str) -> Optional[int]:
        """
        FIXED: Extract year with hex string detection
        """
        # Remove UUIDs and long hex strings FIRST
        text_clean = re.sub(r'[a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12}', '', text, flags=re.I)
        text_clean = re.sub(r'\b[a-f0-9]{8,}\b', '', text_clean, flags=re.I)
        
        # Strategy 1: Standard 20XX format with validation
        year_match = re.search(r'20([12]\d)', text_clean)
        if year_match:
            year = int(year_match.group(0))
            context = text_clean[max(0, year_match.start()-10):year_match.end()+10]
            if self._is_valid_year_context(year, context):
                return year
        
        # Strategy 2: Fiscal year format (FY25, FY2025)
        fy_match = re.search(r'fy\s*(\d{2,4})', text_clean, re.I)
        if fy_match:
            fy_year = fy_match.group(1)
            if len(fy_year) == 2:
                year = int(fy_year)
                full_year = 2000 + year
                if 2020 <= full_year <= self.current_year + 2:
                    return full_year
            elif len(fy_year) == 4:
                year = int(fy_year)
                if 2020 <= year <= self.current_year + 2:
                    return year
        
        # Strategy 3: Extract from URL path (/2025/)
        url_year = re.search(r'/(\d{4})/', text_clean)
        if url_year:
            year = int(url_year.group(1))
            context = text_clean[max(0, url_year.start()-10):url_year.end()+10]
            if self._is_valid_year_context(year, context):
                return year
        
        # Strategy 4: Year in filename (report-2025.pdf)
        filename_year = re.search(r'[-_](\d{4})[.-_]', text_clean)
        if filename_year:
            year = int(filename_year.group(1))
            context = text_clean[max(0, filename_year.start()-10):filename_year.end()+10]
            if self._is_valid_year_context(year, context):
                return year
        
        # Strategy 5: SEC filing numbers (25-000063 = 2025)
        sec_filing = re.search(r'(\d{2})-\d{6}', text_clean)
        if sec_filing:
            filing_year = int(sec_filing.group(1))
            full_year = 2000 + filing_year
            if 2020 <= full_year <= self.current_year + 2:
                return full_year
        
        # Strategy 6: Short year at end (Q4 23 = 2023)
        short_year_patterns = [
            r'(?:4q|q4|q3|q2|q1)\s*(\d{2})\b',
            r'\b(\d{2})(?=\s*$)',
        ]
        for pattern in short_year_patterns:
            short_match = re.search(pattern, text_clean, re.I)
            if short_match:
                year_num = int(short_match.group(1))
                if 15 <= year_num <= 30:
                    full_year = 2000 + year_num
                    if 2020 <= full_year <= self.current_year + 2:
                        return full_year
        
        return None
    
    def _extract_quarter(self, text: str) -> Optional[int]:
        """Extract quarter from text"""
        quarter_match = re.search(r'[q\-]([1-4])(?:\s|q|$)', text, re.I)
        return int(quarter_match.group(1)) if quarter_match else None
    
    def _get_extension(self, url: str) -> str:
        """Get file extension"""
        url_lower = url.lower()
        for ext in self.file_extensions:
            if ext in url_lower:
                return ext
        return '.unknown'
    
    def close(self):
        """Close WebDriver"""
        if self.driver:
            try:
                self.driver.quit()
            except Exception:
                pass
            self.driver = None


# ============================================================================
# MAIN EXECUTION FUNCTIONS
# ============================================================================

def extract_all_companies(input_json_path: str, output_json_path: str = None,
                         start_index: int = 0, end_index: int = None,
                         debug: bool = True) -> Dict[str, Any]:
    """Extract documents for all companies"""
    print("\n" + "="*80)
    print("IR DOCUMENT EXTRACTION - ALL FIXES APPLIED")
    print("✅ JS redirects • Dynamic content • Fixed year extraction • Latest quarterly ID")
    print("="*80)
    
    with open(input_json_path, 'r') as f:
        companies = json.load(f)
    
    if end_index is not None:
        companies = companies[start_index:end_index]
    else:
        companies = companies[start_index:]
    
    print(f"Processing {len(companies)} companies (index {start_index} to {start_index + len(companies)})")
    
    extractor = IRDocumentExtractor(debug=debug)
    all_documents = {}
    latest_quarterly_reports = {}  # NEW: Track latest quarterly per company
    
    try:
        for i, company in enumerate(companies, 1):
            ticker = company.get('ticker', 'UNKNOWN')
            company_name = company.get('company_name', 'Unknown Company')
            ir_url = company.get('investor_relations_url', '')
            
            if not ir_url:
                logger.warning(f"No IR URL for {ticker}, skipping...")
                all_documents[ticker] = []
                continue
            
            print(f"\n[{i}/{len(companies)}] {ticker}")
            
            try:
                documents = extractor.extract_documents(ir_url, ticker, company_name)
                
                for doc in documents:
                    doc.metadata['ticker'] = ticker
                    doc.metadata['company'] = company_name
                
                all_documents[ticker] = documents
                
                # NEW: Identify latest quarterly
                latest_q = extractor.identify_latest_quarterly_report(documents)
                if latest_q:
                    latest_quarterly_reports[ticker] = latest_q
                
                time.sleep(random.uniform(2, 4))
                
            except Exception as e:
                logger.error(f"Error processing {ticker}: {e}")
                all_documents[ticker] = []
                continue
    
    finally:
        extractor.close()
    
    # Print summary
    total = sum(len(docs) for docs in all_documents.values())
    with_docs = sum(1 for docs in all_documents.values() if docs)
    total_pdfs = sum(sum(1 for d in docs if '.pdf' in d.file_extension) 
                     for docs in all_documents.values())
    total_excel = sum(sum(1 for d in docs if '.xls' in d.file_extension) 
                      for docs in all_documents.values())
    old_docs = sum(sum(1 for d in docs if d.extracted_year and d.extracted_year < datetime.now().year - 1) 
                   for docs in all_documents.values())
    
    print("\n" + "="*80)
    print("EXTRACTION SUMMARY")
    print("="*80)
    print(f"✅ Total unique documents: {total}")
    print(f"✅ PDF files: {total_pdfs}")
    print(f"✅ Excel files: {total_excel}")
    print(f"✅ Companies with documents: {with_docs}/{len(companies)}")
    print(f"📊 Latest quarterly reports found: {len(latest_quarterly_reports)}")
    print(f"⚠️ Companies with NO documents: {len(companies) - with_docs}")
    print(f"🚨 Documents with old years (<2024): {old_docs}")
    if companies:
        print(f"✅ Average per company: {total/len(companies):.1f}")
    
    if output_json_path:
        # NEW: Two-tier output structure
        export_data = {
            'all_documents': {},
            'latest_quarterly_reports': {}
        }
        
        for ticker, documents in all_documents.items():
            export_data['all_documents'][ticker] = [asdict(doc) for doc in documents]
        
        for ticker, doc in latest_quarterly_reports.items():
            export_data['latest_quarterly_reports'][ticker] = asdict(doc)
        
        with open(output_json_path, 'w') as f:
            json.dump(export_data, f, indent=2, default=str)
        
        print(f"\n✅ Exported to {output_json_path}")
        print(f"   - All documents: {total}")
        print(f"   - Latest quarterly: {len(latest_quarterly_reports)}")
    
    return {
        'all_documents': all_documents,
        'latest_quarterly_reports': latest_quarterly_reports
    }


def extract_in_batches(input_json_path: str, output_dir: str = "output",
                      batch_size: int = 10, debug: bool = True):
    """Extract documents in batches"""
    with open(input_json_path, 'r') as f:
        companies = json.load(f)
    
    total_companies = len(companies)
    num_batches = (total_companies + batch_size - 1) // batch_size
    
    print(f"\n{'='*80}")
    print(f"BATCH EXTRACTION: {total_companies} companies in {num_batches} batches")
    print(f"{'='*80}")
    
    os.makedirs(output_dir, exist_ok=True)
    all_results = {'all_documents': {}, 'latest_quarterly_reports': {}}
    
    for batch_num in range(num_batches):
        start_idx = batch_num * batch_size
        end_idx = min(start_idx + batch_size, total_companies)
        
        print(f"\n{'='*80}")
        print(f"BATCH {batch_num + 1}/{num_batches}: Companies {start_idx} to {end_idx}")
        print(f"{'='*80}")
        
        batch_output = os.path.join(output_dir, f"batch_{batch_num + 1:03d}.json")
        
        batch_results = extract_all_companies(
            input_json_path,
            output_json_path=batch_output,
            start_index=start_idx,
            end_index=end_idx,
            debug=debug
        )
        
        all_results['all_documents'].update(batch_results['all_documents'])
        all_results['latest_quarterly_reports'].update(batch_results['latest_quarterly_reports'])
        
        print(f"\n✅ Batch {batch_num + 1} complete. Saved to {batch_output}")
    
    combined_output = os.path.join(output_dir, "all_companies_combined.json")
    
    export_data = {
        'all_documents': {},
        'latest_quarterly_reports': {}
    }
    
    for ticker, documents in all_results['all_documents'].items():
        export_data['all_documents'][ticker] = [asdict(doc) for doc in documents]
    
    for ticker, doc in all_results['latest_quarterly_reports'].items():
        export_data['latest_quarterly_reports'][ticker] = asdict(doc)
    
    with open(combined_output, 'w') as f:
        json.dump(export_data, f, indent=2, default=str)
    
    print(f"\n{'='*80}")
    print(f"✅ ALL BATCHES COMPLETE")
    print(f"✅ Combined results saved to {combined_output}")
    print(f"{'='*80}")
    
    return all_results


# ============================================================================
# USAGE EXAMPLES
# ============================================================================



# Uncomment to run full extraction:
# all_results = extract_all_companies(
#     input_json_path='cnbc_companies_with_ir.json',
#     output_json_path='all_documents_fixed.json',
#     debug=True
# )

# Uncomment for batch processing:
# batch_results = extract_in_batches(
#     input_json_path='cnbc_companies_with_ir.json',
#     output_dir='fixed_output',
#     batch_size=20,
#     debug=True
# )

In [51]:
all_results = extract_all_companies(
    input_json_path='../data/catalogue/cnbc_companies_with_ir.json',
    output_json_path='../data/documents/all_documents_fixed.json',
    debug=True
 )

2025-10-09 16:54:09,399 - 
[AMGN] AMGN
2025-10-09 16:54:09,399 - URL: http://investors.amgen.com/
2025-10-09 16:54:09,400 - Setting up ChromeDriver...
2025-10-09 16:54:09,401 - ====== WebDriver manager ======



IR DOCUMENT EXTRACTION - ALL FIXES APPLIED
✅ JS redirects • Dynamic content • Fixed year extraction • Latest quarterly ID
Processing 30 companies (index 0 to 30)

[1/30] AMGN


2025-10-09 16:54:09,659 - Get LATEST chromedriver version for google-chrome
2025-10-09 16:54:09,867 - Get LATEST chromedriver version for google-chrome
2025-10-09 16:54:09,979 - Driver [/Users/RiyanshiKedia/.wdm/drivers/chromedriver/mac64/141.0.7390.65/chromedriver-mac-arm64/chromedriver] found in cache
2025-10-09 16:54:10,793 - ChromeDriver ready
2025-10-09 16:54:19,262 - Discovered sections (priority order):
2025-10-09 16:54:19,263 -   • Quarterly Earnings
2025-10-09 16:54:19,263 -   • SEC Filings
2025-10-09 16:54:19,264 -   • Annual Reports
2025-10-09 16:54:19,264 -   • Amgen Q3 2025 Earnings Conference Call
2025-10-09 16:54:19,264 -   • Tax Forms
2025-10-09 16:54:19,265 -   • Reconciliations
2025-10-09 16:54:19,265 -   • Amgen Green Financing Framework 2022
2025-10-09 16:54:19,265 -   • Amgen Q2 2025 Earnings Conference Call
2025-10-09 16:54:19,266 -   • Q2 2025 Earnings Presentation
2025-10-09 16:54:19,267 -   • Press Releases
2025-10-09 16:54:19,267 - Found 15 main sections
2025-


[2/30] AMZN


2025-10-09 16:57:45,146 - Discovered sections (priority order):
2025-10-09 16:57:45,147 -   • Investor Relations
2025-10-09 16:57:45,147 -   • Annual reports, proxies and shareholder letters
2025-10-09 16:57:45,148 -   • Quarterly results
2025-10-09 16:57:45,148 -   • SEC filings
2025-10-09 16:57:45,149 -   • Investors
2025-10-09 16:57:45,149 -   • Events
2025-10-09 16:57:45,150 - Found 6 main sections
2025-10-09 16:57:45,150 - 
Extracting from section: Investor Relations
2025-10-09 16:57:55,029 -   Found 1 documents
2025-10-09 16:57:55,030 - 
Extracting from section: Annual reports, proxies and shareholder letters
2025-10-09 16:58:04,130 -   Found 87 documents
2025-10-09 16:58:04,130 - 
Extracting from section: Quarterly results
2025-10-09 16:58:13,435 -   Found 79 documents
2025-10-09 16:58:13,436 - 
Extracting from section: SEC filings
2025-10-09 16:58:22,942 -   Found 187 documents
2025-10-09 16:58:22,942 - 
Extracting from section: Investors
2025-10-09 16:58:31,801 -   Found 0 doc

...


[29/30] NKE


2025-10-09 18:23:55,247 - Discovered sections (priority order):
2025-10-09 18:23:55,247 -   • Quarterly Earnings
2025-10-09 18:23:55,248 -   • NYSE NKE  $68.06 -1.03
2025-10-09 18:23:55,248 -   • Investor News
2025-10-09 18:23:55,249 -   • Direct Investment
2025-10-09 18:23:55,249 -   • Board of Directors
2025-10-09 18:23:55,249 -   • Learn More
2025-10-09 18:23:55,250 - Found 6 main sections
2025-10-09 18:23:55,251 - 
Extracting from section: Quarterly Earnings
2025-10-09 18:24:05,687 -   Found 180 documents
2025-10-09 18:24:05,687 - 
Extracting from section: NYSE NKE  $68.06 -1.03
2025-10-09 18:24:24,581 -   Found 0 documents
2025-10-09 18:24:24,582 - 
Extracting from section: Investor News
2025-10-09 18:24:34,196 -   Found 0 documents
2025-10-09 18:24:34,197 - 
Extracting from section: Direct Investment
2025-10-09 18:24:44,909 -   Found 3 documents
2025-10-09 18:24:44,909 - 
Extracting from section: Board of Directors
2025-10-09 18:24:55,280 -   Found 9 documents
2025-10-09 18:24:55


[30/30] PG


2025-10-09 18:25:16,523 - Discovered sections (priority order):
2025-10-09 18:25:16,524 -   • Annual Reports
2025-10-09 18:25:16,524 -   • SEC Filings
2025-10-09 18:25:16,524 -   • Financials
2025-10-09 18:25:16,525 -   • Overview
2025-10-09 18:25:16,525 -   • About P&G
2025-10-09 18:25:16,525 -   • Company Strategy
2025-10-09 18:25:16,525 -   • News
2025-10-09 18:25:16,526 -   • Events & Presentations
2025-10-09 18:25:16,526 -   • Stock Quote
2025-10-09 18:25:16,527 -   • Dividend History
2025-10-09 18:25:16,528 - Found 14 main sections
2025-10-09 18:25:16,528 - 
Extracting from section: Annual Reports
2025-10-09 18:25:29,572 -   Found 29 documents
2025-10-09 18:25:29,577 - 
Extracting from section: SEC Filings
2025-10-09 18:25:40,899 -   Found 310 documents
2025-10-09 18:25:40,900 - 
Extracting from section: Financials
2025-10-09 18:25:52,681 -   Found 1 documents
2025-10-09 18:25:52,682 - 
Extracting from section: Overview
2025-10-09 18:26:05,267 -   Found 0 documents
2025-10-09 18:


EXTRACTION SUMMARY
✅ Total unique documents: 194
✅ PDF files: 153
✅ Excel files: 19
✅ Companies with documents: 24/30
📊 Latest quarterly reports found: 23
⚠️ Companies with NO documents: 6
🚨 Documents with old years (<2024): 20
✅ Average per company: 6.5

✅ Exported to ../data/documents/all_documents_fixed.json
   - All documents: 194
   - Latest quarterly: 23


In [ ]:
# ============================================================================
# DOCUMENT DOWNLOADER: Save all files in company-wise folder structure
# ============================================================================

import os
import json
import time
import random
import requests
from pathlib import Path
from typing import Dict, List, Any
from urllib.parse import urlparse, unquote
import logging
from dataclasses import asdict

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


class DocumentDownloader:
    """Download and organize documents in company-wise folders"""
    
    def __init__(self, base_output_dir: str = "data/reports", 
                 request_delay: tuple = (1, 3),
                 timeout: int = 60,
                 max_retries: int = 3):
        """
        Initialize downloader
        
        Args:
            base_output_dir: Root directory for all downloads (e.g., "data/reports")
            request_delay: (min, max) seconds between downloads
            timeout: Download timeout in seconds
            max_retries: Maximum download retry attempts
        """
        self.base_output_dir = Path(base_output_dir)
        self.request_delay = request_delay
        self.timeout = timeout
        self.max_retries = max_retries
        
        # Create base directory
        self.base_output_dir.mkdir(parents=True, exist_ok=True)
        
        # Download statistics
        self.stats = {
            'total_companies': 0,
            'total_documents': 0,
            'successful_downloads': 0,
            'failed_downloads': 0,
            'skipped_existing': 0,
            'total_size_mb': 0
        }
    
    def sanitize_filename(self, filename: str, max_length: int = 150) -> str:
        """
        Clean filename for safe file system storage
        
        Args:
            filename: Original filename
            max_length: Maximum filename length
            
        Returns:
            Sanitized filename
        """
        # Remove invalid characters
        invalid_chars = '<>:"/\\|?*'
        for char in invalid_chars:
            filename = filename.replace(char, '_')
        
        # Remove leading/trailing spaces and dots
        filename = filename.strip('. ')
        
        # Replace multiple spaces with single space
        filename = ' '.join(filename.split())
        
        # Truncate if too long (preserve extension)
        if len(filename) > max_length:
            name, ext = os.path.splitext(filename)
            filename = name[:max_length - len(ext)] + ext
        
        return filename or 'document'
    
    def sanitize_company_name(self, company_name: str) -> str:
        """
        Clean company name for folder creation
        
        Args:
            company_name: Original company name
            
        Returns:
            Sanitized company name
        """
        # Remove special characters but keep spaces and common punctuation
        name = company_name.replace('/', '_').replace('\\', '_')
        name = ''.join(c for c in name if c.isalnum() or c in ' .-_&')
        name = ' '.join(name.split())  # Normalize spaces
        return name.strip() or 'Unknown_Company'
    
    def get_file_extension(self, url: str, content_type: str = None) -> str:
        """
        Determine file extension from URL or content type
        
        Args:
            url: File URL
            content_type: HTTP content-type header
            
        Returns:
            File extension with dot (e.g., '.pdf')
        """
        # First try URL
        parsed = urlparse(url)
        path = unquote(parsed.path)
        _, ext = os.path.splitext(path)
        
        if ext and len(ext) <= 6:  # Valid extension
            return ext.lower()
        
        # Fallback to content-type
        if content_type:
            content_type_map = {
                'application/pdf': '.pdf',
                'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet': '.xlsx',
                'application/vnd.ms-excel': '.xls',
                'application/vnd.openxmlformats-officedocument.presentationml.presentation': '.pptx',
                'application/vnd.ms-powerpoint': '.ppt',
                'audio/mpeg': '.mp3',
                'video/mp4': '.mp4',
                'text/csv': '.csv'
            }
            return content_type_map.get(content_type.split(';')[0].strip(), '.bin')
        
        return '.bin'
    
    def generate_filename(self, doc: Dict[str, Any], url: str) -> str:
        """
        Generate meaningful filename from document metadata
        
        Args:
            doc: Document information dictionary
            url: Document URL
            
        Returns:
            Generated filename
        """
        # Start with document type
        doc_type = doc.get('document_type', 'Document')
        title = doc.get('title', '')
        year = doc.get('extracted_year')
        quarter = doc.get('extracted_quarter')
        
        # Build filename parts
        parts = [doc_type]
        
        # Add year and quarter if available
        if year:
            if quarter:
                parts.append(f"{year}_Q{quarter}")
            else:
                parts.append(str(year))
        
        # Add title snippet if meaningful
        if title and len(title) > 10:
            # Take first 50 chars of title
            title_clean = self.sanitize_filename(title[:50])
            if title_clean not in doc_type:  # Avoid redundancy
                parts.append(title_clean)
        
        # Join parts
        filename = ' - '.join(parts)
        
        # Add extension
        ext = doc.get('file_extension', self.get_file_extension(url))
        if not filename.endswith(ext):
            filename += ext
        
        return self.sanitize_filename(filename)
    
    def download_file(self, url: str, output_path: Path) -> bool:
        """
        Download single file with retry logic
        
        Args:
            url: File URL
            output_path: Destination file path
            
        Returns:
            True if successful, False otherwise
        """
        for attempt in range(self.max_retries):
            try:
                # Set headers to mimic browser
                headers = {
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
                    'Accept': '*/*',
                    'Accept-Language': 'en-US,en;q=0.9',
                    'Referer': url
                }
                
                # Stream download
                response = requests.get(url, headers=headers, timeout=self.timeout, stream=True)
                response.raise_for_status()
                
                # Get file size
                total_size = int(response.headers.get('content-length', 0))
                
                # Write to file
                with open(output_path, 'wb') as f:
                    if total_size == 0:
                        f.write(response.content)
                    else:
                        downloaded = 0
                        for chunk in response.iter_content(chunk_size=8192):
                            if chunk:
                                f.write(chunk)
                                downloaded += len(chunk)
                
                # Update statistics
                file_size_mb = output_path.stat().st_size / (1024 * 1024)
                self.stats['total_size_mb'] += file_size_mb
                
                logger.info(f"✅ Downloaded: {output_path.name} ({file_size_mb:.2f} MB)")
                return True
                
            except requests.exceptions.RequestException as e:
                logger.warning(f"⚠️ Attempt {attempt + 1}/{self.max_retries} failed: {str(e)[:100]}")
                if attempt < self.max_retries - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
                else:
                    logger.error(f"❌ Failed to download: {url}")
                    return False
            except Exception as e:
                logger.error(f"❌ Unexpected error: {e}")
                return False
        
        return False
    
    def download_company_documents(self, ticker: str, company_name: str, 
                                   documents: List[Dict[str, Any]], 
                                   skip_existing: bool = True) -> Dict[str, Any]:
        """
        Download all documents for a single company
        
        Args:
            ticker: Company ticker symbol
            company_name: Company name
            documents: List of document dictionaries
            skip_existing: Skip files that already exist
            
        Returns:
            Download statistics for this company
        """
        if not documents:
            logger.info(f"⏭️ {ticker}: No documents to download")
            return {'downloaded': 0, 'failed': 0, 'skipped': 0}
        
        # Create company folder
        company_folder_name = f"{ticker} - {self.sanitize_company_name(company_name)}"
        company_dir = self.base_output_dir / company_folder_name
        company_dir.mkdir(parents=True, exist_ok=True)
        
        logger.info(f"\n{'='*80}")
        logger.info(f"📁 {ticker}: {company_name}")
        logger.info(f"   Folder: {company_dir}")
        logger.info(f"   Documents to download: {len(documents)}")
        logger.info(f"{'='*80}")
        
        company_stats = {'downloaded': 0, 'failed': 0, 'skipped': 0}
        
        for i, doc in enumerate(documents, 1):
            try:
                url = doc.get('url')
                if not url:
                    logger.warning(f"⚠️ [{i}/{len(documents)}] No URL found")
                    company_stats['failed'] += 1
                    continue
                
                # Generate filename
                filename = self.generate_filename(doc, url)
                output_path = company_dir / filename
                
                # Check if file exists
                if skip_existing and output_path.exists():
                    logger.info(f"⏭️ [{i}/{len(documents)}] Already exists: {filename}")
                    company_stats['skipped'] += 1
                    self.stats['skipped_existing'] += 1
                    continue
                
                # Download file
                logger.info(f"⬇️ [{i}/{len(documents)}] Downloading: {filename}")
                success = self.download_file(url, output_path)
                
                if success:
                    company_stats['downloaded'] += 1
                    self.stats['successful_downloads'] += 1
                else:
                    company_stats['failed'] += 1
                    self.stats['failed_downloads'] += 1
                
                # Delay between downloads
                if i < len(documents):
                    delay = random.uniform(*self.request_delay)
                    time.sleep(delay)
                
            except Exception as e:
                logger.error(f"❌ Error processing document: {e}")
                company_stats['failed'] += 1
                self.stats['failed_downloads'] += 1
        
        # Save metadata JSON for this company
        metadata_path = company_dir / "_metadata.json"
        with open(metadata_path, 'w') as f:
            json.dump({
                'ticker': ticker,
                'company_name': company_name,
                'total_documents': len(documents),
                'documents': documents,
                'download_stats': company_stats,
                'downloaded_at': time.strftime('%Y-%m-%d %H:%M:%S')
            }, f, indent=2)
        
        logger.info(f"\n✅ {ticker} Complete: {company_stats['downloaded']} downloaded, "
                   f"{company_stats['failed']} failed, {company_stats['skipped']} skipped")
        
        return company_stats
    
    def download_all_companies(self, input_json_path: str, 
                              companies_to_process: List[str] = None,
                              skip_existing: bool = True) -> None:
        """
        Download documents for all companies from JSON file
        
        Args:
            input_json_path: Path to JSON file with extraction results
            companies_to_process: List of tickers to process (None = all)
            skip_existing: Skip files that already exist
        """
        logger.info("\n" + "="*80)
        logger.info("📥 DOCUMENT DOWNLOADER - Company-wise folder structure")
        logger.info(f"📂 Output directory: {self.base_output_dir.absolute()}")
        logger.info("="*80)
        
        # Load extraction results
        with open(input_json_path, 'r') as f:
            data = json.load(f)
        
        # Handle both single-tier and two-tier structures
        if 'all_documents' in data:
            all_documents = data['all_documents']
        else:
            all_documents = data
        
        # Filter companies if specified
        if companies_to_process:
            all_documents = {k: v for k, v in all_documents.items() 
                           if k in companies_to_process}
        
        self.stats['total_companies'] = len(all_documents)
        self.stats['total_documents'] = sum(len(docs) for docs in all_documents.values())
        
        logger.info(f"\n📊 Processing {self.stats['total_companies']} companies")
        logger.info(f"📊 Total documents: {self.stats['total_documents']}")
        
        # Process each company
        for company_num, (ticker, documents) in enumerate(all_documents.items(), 1):
            logger.info(f"\n[{company_num}/{self.stats['total_companies']}] Processing {ticker}...")
            
            # Get company name from first document
            company_name = 'Unknown Company'
            if documents and len(documents) > 0:
                company_name = documents[0].get('metadata', {}).get('company', company_name)
            
            self.download_company_documents(ticker, company_name, documents, skip_existing)
        
        # Print final summary
        self.print_summary()
    
    def print_summary(self) -> None:
        """Print download summary statistics"""
        logger.info("\n" + "="*80)
        logger.info("📊 DOWNLOAD SUMMARY")
        logger.info("="*80)
        logger.info(f"✅ Total companies processed: {self.stats['total_companies']}")
        logger.info(f"✅ Total documents: {self.stats['total_documents']}")
        logger.info(f"✅ Successfully downloaded: {self.stats['successful_downloads']}")
        logger.info(f"⏭️ Skipped (already exist): {self.stats['skipped_existing']}")
        logger.info(f"❌ Failed downloads: {self.stats['failed_downloads']}")
        logger.info(f"💾 Total size: {self.stats['total_size_mb']:.2f} MB")
        logger.info(f"📂 Output directory: {self.base_output_dir.absolute()}")
        logger.info("="*80)
        
        if self.stats['successful_downloads'] > 0:
            success_rate = (self.stats['successful_downloads'] / 
                          (self.stats['successful_downloads'] + self.stats['failed_downloads'])) * 100
            logger.info(f"✅ Success rate: {success_rate:.1f}%")


# ============================================================================
# USAGE EXAMPLES
# ============================================================================

def main():
    """Main execution function"""
    
    # Example 1: Download ALL documents for ALL companies
    downloader = DocumentDownloader(
        base_output_dir="../data/reports",  # This creates: data/reports/AAPL - Apple Inc/...
        request_delay=(1, 3),             # Wait 1-3 seconds between downloads
        timeout=60,                        # 60 second timeout per file
        max_retries=3                      # Retry 3 times on failure
    )
    
    downloader.download_all_companies(
        input_json_path='../data/documents/all_documents_fixed.json',  # Your extraction output
        skip_existing=True  # Skip files that already exist
    )
    
    
    # Example 2: Download only specific companies
    """
    downloader = DocumentDownloader(base_output_dir="data/reports")
    downloader.download_all_companies(
        input_json_path='all_documents_fixed.json',
        companies_to_process=['AAPL', 'MSFT', 'GOOGL'],  # Only these tickers
        skip_existing=True
    )
    """
    
    
    # Example 3: Resume interrupted download
    """
    # If download was interrupted, just run again with skip_existing=True
    # It will skip files that were already downloaded and continue from where it stopped
    downloader = DocumentDownloader(base_output_dir="data/reports")
    downloader.download_all_companies(
        input_json_path='all_documents_fixed.json',
        skip_existing=True  # Will skip already downloaded files
    )
    """


if __name__ == "__main__":
    main()

2025-10-09 18:35:36,575 - 
2025-10-09 18:35:36,575 - 📥 DOCUMENT DOWNLOADER - Company-wise folder structure
2025-10-09 18:35:36,576 - 📂 Output directory: /Users/RiyanshiKedia/Documents/GitHub/investment-report-extractor/notebooks/../data/reports
2025-10-09 18:35:36,576 - ================================================================================
2025-10-09 18:35:36,578 - 
📊 Processing 30 companies
2025-10-09 18:35:36,579 - 📊 Total documents: 194
2025-10-09 18:35:36,579 - 
[1/30] Processing AMGN...
2025-10-09 18:35:36,580 - 
2025-10-09 18:35:36,580 - 📁 AMGN: AMGN
2025-10-09 18:35:36,580 -    Folder: ../data/reports/AMGN - AMGN
2025-10-09 18:35:36,580 -    Documents to download: 8
2025-10-09 18:35:36,581 - ================================================================================
2025-10-09 18:35:36,581 - ⬇️ [1/8] Downloading: Financial Document - 2025 - 0000932471-25-001153.rtf.unknown
2025-10-09 18:35:38,208 - ✅ Downloaded: Financial Document - 2025 - 0000932471-25-001153.rtf